## Phase 2 – PCA, Logistic Regression & Model Export

> **Note:** Cells below continue from the Phase 1 notebook. Run Phase 1 first so `df_heart` is in memory, OR run the standalone re-load cell below.

### 2.1 – Re-load & Prep UCI Heart Data (all 13 features)

In [ ]:
import pandas as pd
import numpy as np

# ── Column names from UCI documentation
columns = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak',
    'slope', 'ca', 'thal', 'target'
]

# Load raw file (adjust path if running in Colab from Drive)
try:
    df_heart = pd.read_csv(
        '/content/drive/MyDrive/Colab Notebooks/608 PROJECT/Input Files /processed.cleveland.data',
        names=columns
    )
except FileNotFoundError:
    # Fallback: load from current directory
    df_heart = pd.read_csv('processed_cleveland.data', names=columns)

# Clean
df_heart.replace('?', pd.NA, inplace=True)
df_heart = df_heart.apply(pd.to_numeric, errors='coerce')
df_heart.fillna(df_heart.median(numeric_only=True), inplace=True)
df_heart['target'] = df_heart['target'].apply(lambda x: 1 if x > 0 else 0)

print(f"Shape: {df_heart.shape}")
print(f"Label balance - 0: {(df_heart['target']==0).sum()}, 1: {(df_heart['target']==1).sum()}")
df_heart.head()


### 2.2 – Feature Selection & Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# All 13 clinical features (no metadata cols)
FEATURE_COLS = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
                'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
TARGET_COL   = 'target'

X = df_heart[FEATURE_COLS].values
y = df_heart[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardise — CRITICAL: fit only on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")


### 2.3 – PCA: Reduce to 'Readiness Vector'

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# First, fit PCA on all components to see explained variance
pca_full = PCA(random_state=42)
pca_full.fit(X_train_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cumvar)+1), cumvar, marker='o', color='steelblue')
plt.axhline(0.90, linestyle='--', color='red', label='90% threshold')
plt.axhline(0.95, linestyle='--', color='orange', label='95% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA – Explained Variance by Component Count')
plt.legend()
plt.tight_layout()
plt.show()

# Identify how many components hit 90% / 95%
n_90 = np.argmax(cumvar >= 0.90) + 1
n_95 = np.argmax(cumvar >= 0.95) + 1
print(f"Components for 90% variance: {n_90}")
print(f"Components for 95% variance: {n_95}")


In [ ]:
# Choose N_COMPONENTS based on scree plot above (typically 6-8 for Cleveland)
N_COMPONENTS = n_90  # Use 90% threshold — good balance of compression vs. info

pca = PCA(n_components=N_COMPONENTS, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f"PCA reduced features: {X_train_scaled.shape[1]} → {X_train_pca.shape[1]}")
print(f"Total variance retained: {pca.explained_variance_ratio_.sum():.4f}")

# Component loadings heatmap
import seaborn as sns

loadings = pd.DataFrame(
    pca.components_.T,
    index=FEATURE_COLS,
    columns=[f'PC{i+1}' for i in range(N_COMPONENTS)]
)

plt.figure(figsize=(10, 5))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('PCA Component Loadings — Feature Contributions')
plt.tight_layout()
plt.show()


### 2.4 – Logistic Regression Classifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, RocCurveDisplay, f1_score)

lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_pca, y_train)

y_pred  = lr.predict(X_test_pca)
y_proba = lr.predict_proba(X_test_pca)[:, 1]

print("=" * 50)
print("Classification Report")
print("=" * 50)
print(classification_report(y_test, y_pred, target_names=['No Risk (0)', 'At Risk (1)']))

f1  = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
print(f"F1 Score : {f1:.4f}   (target ≥ 0.75)")
print(f"ROC-AUC  : {auc:.4f}")


In [ ]:
# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Risk','At Risk'],
            yticklabels=['No Risk','At Risk'], ax=axes[0])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('Actual')
axes[0].set_xlabel('Predicted')

# ROC curve
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1])
axes[1].set_title(f'ROC Curve (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--')

plt.tight_layout()
plt.show()


### 2.5 – Optional: Cross-Validated Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs', 'liblinear'],
    'penalty': ['l2']
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(
    LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1
)
grid.fit(X_train_pca, y_train)

print(f"Best params : {grid.best_params_}")
print(f"Best CV F1  : {grid.best_score_:.4f}")

# Re-evaluate best model on held-out test set
best_lr = grid.best_estimator_
y_pred_best  = best_lr.predict(X_test_pca)
y_proba_best = best_lr.predict_proba(X_test_pca)[:, 1]
print("\nTest set results with best model:")
print(classification_report(y_test, y_pred_best, target_names=['No Risk (0)', 'At Risk (1)']))
print(f"F1 : {f1_score(y_test, y_pred_best):.4f}")
print(f"AUC: {roc_auc_score(y_test, y_proba_best):.4f}")

# Use best model going forward
lr = best_lr


### 2.6 – Save Model Artifacts (scaler + pca + model)

In [ ]:
import joblib, os

ARTIFACT_DIR = './model_artifacts'
os.makedirs(ARTIFACT_DIR, exist_ok=True)

joblib.dump(scaler,       f'{ARTIFACT_DIR}/scaler.pkl')
joblib.dump(pca,          f'{ARTIFACT_DIR}/pca.pkl')
joblib.dump(lr,           f'{ARTIFACT_DIR}/logistic_model.pkl')

# Also save metadata so the inference script knows the feature order
import json
metadata = {
    'feature_cols': FEATURE_COLS,
    'n_pca_components': N_COMPONENTS,
    'pca_variance_retained': float(pca.explained_variance_ratio_.sum()),
    'label_map': {'0': 'No Cardiovascular Risk', '1': 'Cardiovascular Risk Detected'},
    'model_type': 'LogisticRegression + PCA',
    'trained_on': 'UCI Cleveland Heart Disease Dataset (n=303)',
    'f1_score': float(f1_score(y_test, y_pred_best if "y_pred_best" in dir() else y_pred)),
    'roc_auc': float(roc_auc_score(y_test, y_proba_best if "y_proba_best" in dir() else y_proba))
}
with open(f'{ARTIFACT_DIR}/model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Saved artifacts:")
for f in os.listdir(ARTIFACT_DIR):
    print(f"  {ARTIFACT_DIR}/{f}")


### 2.7 – Upload Model Artifacts to S3 (Results Zone)

In [ ]:
# ── Requires AWS credentials in Colab (set via Secrets or environment vars)
# Uncomment and run once artifacts are validated

# import boto3
# s3 = boto3.client('s3')
# BUCKET = 'pulsepoint-raw-zone-mena'   # update if your results bucket differs
# RESULTS_PREFIX = 'model_artifacts/'

# for fname in os.listdir(ARTIFACT_DIR):
#     local_path = f'{ARTIFACT_DIR}/{fname}'
#     s3_key     = f'{RESULTS_PREFIX}{fname}'
#     s3.upload_file(local_path, BUCKET, s3_key)
#     print(f'Uploaded: s3://{BUCKET}/{s3_key}')

print("S3 upload cell ready — uncomment to push artifacts after local validation.")
